In [1]:
import os
os.environ["GROQ_API_KEY"] = "api key"

In [2]:
!pip install numpy==1.26.4

In [3]:
!pip install -q groq
!pip install -q transformers==4.40.0
!pip install -q datasets==2.18.0
!pip install -q peft==0.10.0
!pip install -q trl==0.8.6
!pip install -q accelerate==0.29.3
!pip install -q bitsandbytes==0.43.1
!pip install -q torch==2.2.2
print("All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
gcsfs 2025

In [4]:
!pip uninstall -y triton
!pip install triton==2.1.0

ERROR: Could not find a version that satisfies the requirement triton==2.1.0 (from versions: 2.2.0, 2.3.0, 2.3.1, 3.0.0, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0, 3.5.1, 3.6.0, 3.7.0)
ERROR: No matching distribution found for triton==2.1.0


In [5]:
import os, gc, json, time, re, random, torch, warnings
warnings.filterwarnings("ignore")
from dataclasses import dataclass, field
from typing import Optional
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY not set.")

client     = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "llama-3.1-8b-instant"

BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FT_DIR       = "./ft_extractor"
TRAIN_FILE   = "./train.json"
VAL_FILE     = "./val.json"
DATA_PATH    = "./project_omni_pclub_secy_recruit_dataset.json"
REPORTS_DIR  = "./reports"

ALLOWED_RELS = {"at", "with", "motive", "owns", "knows", "did"}

MOTIVE_WORDS = {
    "motive", "conflict", "argument", "hate", "dispute",
    "rival",  "bitter",  "disagree", "jealous", "envy",
    "anger",  "resent",  "betray",   "revenge", "grudge",
}

PTS_TIMELINE_LIE      = 25
PTS_WITNESS_CONFLICT  = 20
PTS_SCENE_PRESENCE    = 15
PTS_MOTIVE            = 15
PTS_NO_ALIBI          = 10
PTS_ALIBI_CLEAR       = -15

MAX_ROUNDS   = 3
GAP_RATIO    = 1.5

token_log = {"prompt": 0, "completion": 0, "total": 0}

os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs("outputs",   exist_ok=True)
os.makedirs("logs",      exist_ok=True)


def ask_groq(msgs: list, max_tok: int = 300) -> str:
    global token_log
    for attempt in range(3):
        try:
            time.sleep(1.5)
            r = client.chat.completions.create(
                model=GROQ_MODEL, messages=msgs,
                max_tokens=max_tok, temperature=0.2,
            )
            if not r or not getattr(r, "choices", None):
                return ""
            u = getattr(r, "usage", None)
            if u:
                token_log["prompt"]     += getattr(u, "prompt_tokens",     0)
                token_log["completion"] += getattr(u, "completion_tokens", 0)
                token_log["total"]      += getattr(u, "total_tokens",      0)
            return r.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if "401" in err:
                raise RuntimeError("Bad Groq API key.")
            if "429" in err:
                w = 15 * (attempt + 1)
                print(f"[Groq] rate limit, waiting {w}s...")
                time.sleep(w)
            else:
                print(f"[Groq] {err}")
                if attempt == 2:
                    return ""
    return ""


def clean(text) -> str:
    if isinstance(text, list):
        return " ".join(str(t) for t in text).strip()
    return str(text).strip() if text else ""


print(f"GPU: {torch.cuda.is_available()}")
print("config done.")

GPU: False
config done.


In [6]:
@dataclass
class Claim:
    who       : str
    rel       : str
    what      : str
    when      : Optional[str]
    said_by   : str
    conf      : float
    original  : str

    def ok(self) -> bool:
        return bool(
            self.who.strip() and
            self.rel in ALLOWED_RELS and
            self.what.strip()
        )


@dataclass
class Clash:
    c1          : Claim
    c2          : Claim
    clash_type  : str
    why         : str
    strength    : float = 1.0


@dataclass
class Person:
    name          : str
    claims        : list = field(default_factory=list)
    clashes       : list = field(default_factory=list)
    suspicion     : float = 0.0
    alibi_solid   : bool  = False
    has_motive    : bool  = False
    q_count       : int   = 0


@dataclass
class TimeNode:
    suspect  : str
    time_str : str
    location : str
    source   : str
    time_mins: int = 0


print("classes done.")

classes done.


In [7]:
def parse_time(t: str) -> int:
    t = t.strip().upper()
    try:
        is_pm = "PM" in t
        is_am = "AM" in t
        t = t.replace("AM", "").replace("PM", "").strip()
        if ":" in t:
            h, m = t.split(":")
        else:
            h, m = t, "0"
        h, m = int(h.strip()), int(m.strip())
        if is_pm and h != 12:
            h += 12
        if is_am and h == 12:
            h = 0
        return h * 60 + m
    except:
        return -1


def extract_location(activity: str) -> Optional[str]:
    activity = activity.lower()
    patterns = [
        r"(?:at|in|to|arrives at|reaches|heads to)\s+(the\s+[\w\s]{3,35})",
        r"(?:at|in|to)\s+([\w\s]{3,25}(?:library|room|office|home|hotel|cafe|garden|hall|mansion|university|department))",
        r"(?:returns to|goes to|visits)\s+([\w\s]{3,30})",
    ]
    for pat in patterns:
        m = re.search(pat, activity)
        if m:
            loc = m.group(1).strip()
            if len(loc) > 3:
                return loc[:60]
    generic_locs = ["home", "office", "library", "university", "apartment", "cafe", "hotel"]
    for loc in generic_locs:
        if loc in activity:
            return loc
    return None


class TimelineGraph:
    def __init__(self):
        self.nodes : list[TimeNode] = []

    def load_suspect(self, name: str, profile: dict):
        raw_timeline = profile.get("timeline", {})
        if isinstance(raw_timeline, dict):
            raw_timeline = raw_timeline.get("timeline", [])
        if not isinstance(raw_timeline, list):
            return
        for entry in raw_timeline:
            t_str    = entry.get("time",     "")
            activity = entry.get("activity", "")
            location = extract_location(activity)
            if not location or not t_str:
                continue
            t_mins = parse_time(t_str)
            if t_mins < 0:
                continue
            self.nodes.append(TimeNode(
                suspect  = name,
                time_str = t_str,
                location = location,
                source   = "timeline",
                time_mins= t_mins,
            ))

    def get_location_at(self, suspect: str, time_mins: int,
                         window: int = 45) -> Optional[str]:
        candidates = [
            n for n in self.nodes
            if n.suspect == suspect and
               abs(n.time_mins - time_mins) <= window
        ]
        if not candidates:
            return None
        closest = min(candidates, key=lambda n: abs(n.time_mins - time_mins))
        return closest.location

    def rule1_self_lie(self, suspect: str,
                        claimed_loc: str,
                        claimed_time_mins: int) -> Optional[str]:
        actual = self.get_location_at(suspect, claimed_time_mins)
        if not actual:
            return None
        if actual.lower() not in claimed_loc.lower() and \
           claimed_loc.lower() not in actual.lower():
            return (
                f"{suspect} claims to be at '{claimed_loc}' "
                f"but their timeline places them at '{actual}' "
                f"at the same time."
            )
        return None

    def rule2_witness_conflict(self, suspect: str,
                                witness: str,
                                witness_says_loc: str,
                                time_mins: int) -> Optional[str]:
        actual = self.get_location_at(suspect, time_mins)
        if not actual:
            return None
        if actual.lower() not in witness_says_loc.lower() and \
           witness_says_loc.lower() not in actual.lower():
            return (
                f"{witness} claims to have seen {suspect} at "
                f"'{witness_says_loc}', but {suspect}'s timeline "
                f"places them at '{actual}' at that time."
            )
        return None

    def rule3_scene_presence(self, suspect: str,
                              crime_location: str,
                              crime_time_mins: int,
                              window: int = 60) -> bool:
        loc = self.get_location_at(suspect, crime_time_mins, window)
        if not loc:
            return False
        return (
            crime_location.lower() in loc.lower() or
            loc.lower() in crime_location.lower()
        )

    def run_all_checks(self, suspects: list[dict],
                        crime_location: str,
                        crime_time_str: str) -> list[tuple]:
        flags   = []
        c_time  = parse_time(crime_time_str) if crime_time_str else -1

        for s in suspects:
            name = s["name"]

            if c_time > 0:
                at_scene = self.rule3_scene_presence(
                    name, crime_location, c_time)
                if at_scene:
                    flags.append((name, "scene_presence",
                        f"{name}'s timeline places them at or near "
                        f"the crime scene around the time of the murder.",
                        PTS_SCENE_PRESENCE))

            testimony = clean(s.get("testimony", ""))
            if testimony:
                loc_match = re.search(
                    r"(?:saw|noticed|spotted)\s+([\w\s]+)\s+"
                    r"(?:at|near|in)\s+(the\s+[\w\s]{3,30}|[\w\s]{3,20}"
                    r"(?:section|room|wing|area|hall))",
                    testimony, re.IGNORECASE
                )
                if loc_match:
                    seen_person = loc_match.group(1).strip()
                    seen_loc    = loc_match.group(2).strip()
                    conflict    = self.rule2_witness_conflict(
                        seen_person, name, seen_loc, c_time)
                    if conflict:
                        flags.append((seen_person, "witness_conflict",
                            conflict, PTS_WITNESS_CONFLICT))

        return flags


print("TimelineGraph ready.")

TimelineGraph ready.


In [8]:
def get_location_from_text(text: str) -> Optional[str]:
    pats = [
        r"I was (?:at|in) (the [\w\s]{3,40})",
        r"I was (?:at|in) ([\w\s]{3,30}"
        r"(?:library|room|office|home|hotel|garden|hall))",
        r"(?:at|in) (my [\w\s]{3,20})",
    ]
    for p in pats:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return m.group(1).strip()[:80]
    return None


def build_pairs(dataset: list) -> list:
    pairs = []
    for case in dataset:
        victim = case.get("victim", {}).get("name", "the victim")
        for s in case.get("suspects", []):
            name    = s.get("name", "")
            testy   = clean(s.get("testimony",    ""))
            motive  = clean(s.get("motive",       ""))
            opp     = clean(s.get("opportunity",  ""))
            rel     = clean(s.get("relationship", ""))
            if not testy or not name:
                continue
            facts = []
            if rel:
                facts.append({
                    "who": name, "rel": "knows",
                    "what": victim, "when": None, "conf": 1.0,
                })
            if motive:
                facts.append({
                    "who": name, "rel": "motive",
                    "what": motive[:120], "when": None, "conf": 1.0,
                })
            if opp:
                facts.append({
                    "who": name, "rel": "did",
                    "what": opp[:120], "when": None, "conf": 1.0,
                })
            loc = get_location_from_text(testy)
            if loc:
                facts.append({
                    "who": name, "rel": "at",
                    "what": loc, "when": None, "conf": 1.0,
                })
            if not facts:
                continue
            inp = (
                f"Extract structured facts from this murder "
                f"investigation testimony.\n"
                f"Speaker: {name}\n"
                f"Testimony: {testy[:400]}\n\n"
                f"Return a JSON array. Each item must have: "
                f"who, rel, what, when (or null), conf (0.0-1.0).\n"
                f"rel must be one of: at, with, motive, owns, knows, did.\n"
                f"Return ONLY the JSON array."
            )
            pairs.append({"input": inp, "output": json.dumps(facts)})
    return pairs


with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(f"loaded {len(dataset)} cases.")

all_pairs = build_pairs(dataset)
random.seed(42)
random.shuffle(all_pairs)

cut         = int(0.8 * len(all_pairs))
train_pairs = all_pairs[:cut]
val_pairs   = all_pairs[cut:]

with open(TRAIN_FILE, "w") as f: json.dump(train_pairs, f, indent=2)
with open(VAL_FILE,   "w") as f: json.dump(val_pairs,   f, indent=2)

print(f"train: {len(train_pairs)} | val: {len(val_pairs)}")
print("\nsample input:\n",  train_pairs[0]["input"][:300])
print("\nsample output:\n", train_pairs[0]["output"])

JSONDecodeError: Unterminated string starting at: line 54867 column 37 (char 5242850)

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, BitsAndBytesConfig,
)
from peft    import LoraConfig, get_peft_model, TaskType
from trl     import SFTTrainer
from datasets import Dataset as HFData

with open(TRAIN_FILE) as f: tr = json.load(f)
with open(VAL_FILE)   as f: vl = json.load(f)


def fmt(p: dict) -> str:
    return (
        "<|system|>You extract structured facts from murder investigation "
        "testimonies. Output only valid JSON.</s>\n"
        f"<|user|>{p['input']}</s>\n"
        f"<|assistant|>{p['output']}</s>"
    )


tr_hf = HFData.from_dict({"text": [fmt(p) for p in tr]})
vl_hf = HFData.from_dict({"text": [fmt(p) for p in vl]})
print(f"train: {len(tr_hf)} | val: {len(vl_hf)}")

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("loading TinyLlama...")
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.eos_token

mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto")
mdl.config.use_cache = False

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32,
    lora_dropout=0.05, target_modules=["q_proj", "v_proj"], bias="none",
)
mdl = get_peft_model(mdl, lora)
mdl.print_trainable_parameters()

args = TrainingArguments(
    output_dir=FT_DIR, num_train_epochs=3,
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=2, learning_rate=2e-4,
    fp16=True, logging_steps=25, evaluation_strategy="epoch",
    save_strategy="epoch", load_best_model_at_end=True,
    warmup_ratio=0.05, lr_scheduler_type="cosine", report_to="none",
)

trainer = SFTTrainer(
    model=mdl, args=args, train_dataset=tr_hf, eval_dataset=vl_hf,
    dataset_text_field="text", max_seq_length=512, tokenizer=tok,
)

print("fine-tuning — ~45 mins on T4.")
trainer.train()
trainer.save_model(FT_DIR)
tok.save_pretrained(FT_DIR)
print(f"saved to {FT_DIR}.")

del mdl, trainer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
def extract_location_from_testimony(testimony: str) -> Optional[str]:
    patterns = [
        r"I was (?:at|in) (the [\w\s]{3,40})",
        r"I was (?:at|in) ([\w\s]{3,30}(?:library|room|office|home|hotel|garden|hall))",
        r"(?:at|in) (my [\w\s]{3,20})",
    ]
    for pat in patterns:
        m = re.search(pat, testimony, re.IGNORECASE)
        if m:
            return m.group(1).strip()[:80]
    return None


def build_training_pairs(dataset: list) -> list:
    pairs = []
    for case in dataset:
        victim_name = case.get("victim", {}).get("name", "the victim")
        for suspect in case.get("suspects", []):
            name         = suspect.get("name", "")
            testimony    = normalize(suspect.get("testimony",    ""))
            motive       = normalize(suspect.get("motive",       ""))
            opportunity  = normalize(suspect.get("opportunity",  ""))
            relationship = normalize(suspect.get("relationship", ""))
            if not testimony or not name:
                continue
            facts = []
            if relationship:
                facts.append({
                    "subject": name, "predicate": "knew",
                    "obj": victim_name, "time": None, "confidence": 1.0,
                })
            if motive:
                facts.append({
                    "subject": name, "predicate": "has_motive",
                    "obj": motive[:120], "time": None, "confidence": 1.0,
                })
            if opportunity:
                facts.append({
                    "subject": name, "predicate": "did_action",
                    "obj": opportunity[:120], "time": None, "confidence": 1.0,
                })
            location = extract_location_from_testimony(testimony)
            if location:
                facts.append({
                    "subject": name, "predicate": "was_at",
                    "obj": location, "time": None, "confidence": 1.0,
                })
            if not facts:
                continue
            input_text = (
                f"Extract structured facts from this murder investigation testimony.\n"
                f"Speaker: {name}\n"
                f"Testimony: {testimony[:400]}\n\n"
                f"Return a JSON array. Each item must have: "
                f"subject, predicate, obj, time (or null), confidence (0.0-1.0).\n"
                f"predicate must be one of: was_at, was_with, has_motive, owns, knew, did_action.\n"
                f"Return ONLY the JSON array, no other text."
            )
            pairs.append({"input": input_text, "output": json.dumps(facts)})
    return pairs


with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(f"Loaded {len(dataset)} cases.")

all_pairs = build_training_pairs(dataset)
random.seed(42)
random.shuffle(all_pairs)

split       = int(0.8 * len(all_pairs))
train_pairs = all_pairs[:split]
val_pairs   = all_pairs[split:]

with open(TRAIN_FILE, "w") as f:
    json.dump(train_pairs, f, indent=2)
with open(VAL_FILE, "w") as f:
    json.dump(val_pairs, f, indent=2)

print(f"Training pairs  : {len(train_pairs)}")
print(f"Validation pairs: {len(val_pairs)}")
print("\nSample input:\n",  train_pairs[0]["input"][:300])
print("\nSample output:\n", train_pairs[0]["output"])

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft    import LoraConfig, get_peft_model, TaskType
from trl     import SFTTrainer
from datasets import Dataset as HFDataset

with open(TRAIN_FILE) as f:
    train_raw = json.load(f)
with open(VAL_FILE) as f:
    val_raw = json.load(f)


def format_prompt(pair: dict) -> str:
    return (
        "<|system|>You are a structured fact extractor for murder "
        "investigations. Given a testimony, output only a valid JSON "
        "array of facts.</s>\n"
        f"<|user|>{pair['input']}</s>\n"
        f"<|assistant|>{pair['output']}</s>"
    )


train_hf = HFDataset.from_dict({"text": [format_prompt(p) for p in train_raw]})
val_hf   = HFDataset.from_dict({"text": [format_prompt(p) for p in val_raw]})
print(f"Train: {len(train_hf)} | Val: {len(val_hf)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

print("Loading TinyLlama base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ["q_proj", "v_proj"],
    bias           = "none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir                  = FINETUNE_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 2,
    learning_rate               = 2e-4,
    fp16                        = True,
    logging_steps               = 25,
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    report_to                   = "none",
)

trainer = SFTTrainer(
    model              = model,
    args               = training_args,
    train_dataset      = train_hf,
    eval_dataset       = val_hf,
    dataset_text_field = "text",
    max_seq_length     = 512,
    tokenizer          = tokenizer,
)

print("Starting fine-tuning — approximately 45 minutes on a T4 GPU.")
trainer.train()

trainer.save_model(FINETUNE_DIR)
tokenizer.save_pretrained(FINETUNE_DIR)
print(f"Fine-tuned model saved to {FINETUNE_DIR}.")

del model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from peft import PeftModel

print("Loading fine-tuned extractor...")
FINETUNED_AVAILABLE = False
ft_model     = None
ft_tokenizer = None

try:
    ft_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype = torch.float16,
        device_map  = "auto",
    )
    ft_model = PeftModel.from_pretrained(ft_base, FINETUNE_DIR)
    ft_model.eval()
    ft_tokenizer = AutoTokenizer.from_pretrained(FINETUNE_DIR)
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
    FINETUNED_AVAILABLE = True
    print("Fine-tuned model loaded successfully.")
except Exception as e:
    print(f"Fine-tuned model unavailable: {e}")
    print("Falling back to Groq-only extraction.")


def _parse_json_facts(raw: str, speaker: str, testimony: str) -> list:
    try:
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        start = raw.find("[")
        end   = raw.rfind("]") + 1
        if start == -1 or end == 0:
            return []
        data  = json.loads(raw[start:end])
        facts = []
        for item in data:
            if not isinstance(item, dict):
                continue
            fact = Fact(
                subject    = str(item.get("subject",    "")).strip(),
                predicate  = str(item.get("predicate",  "")).strip(),
                obj        = str(item.get("obj",        "")).strip(),
                time       = item.get("time"),
                source     = speaker,
                confidence = float(item.get("confidence", 0.7)),
                raw        = testimony,
            )
            if fact.is_valid():
                facts.append(fact)
        return facts
    except Exception as e:
        print(f"[Parser] JSON parse error: {e}")
        return []


def extract_with_finetuned(testimony: str, speaker: str) -> list:
    prompt = (
        "<|system|>You are a structured fact extractor for murder "
        "investigations. Given a testimony, output only a valid JSON "
        "array of facts.</s>\n"
        f"<|user|>Extract structured facts from this murder investigation testimony.\n"
        f"Speaker: {speaker}\n"
        f"Testimony: {testimony[:400]}\n\n"
        f"Return a JSON array. Each item must have: "
        f"subject, predicate, obj, time (or null), confidence (0.0-1.0).\n"
        f"predicate must be one of: was_at, was_with, has_motive, owns, knew, did_action.\n"
        f"Return ONLY the JSON array, no other text.</s>\n"
        f"<|assistant|>"
    )
    try:
        inputs = ft_tokenizer(
            prompt,
            return_tensors = "pt",
            truncation     = True,
            max_length     = 512,
        ).to(ft_model.device)
        with torch.no_grad():
            outputs = ft_model.generate(
                **inputs,
                max_new_tokens = 256,
                temperature    = 0.1,
                do_sample      = True,
                pad_token_id   = ft_tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        raw = ft_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return _parse_json_facts(raw, speaker, testimony)
    except Exception as e:
        print(f"[FinetunedExtractor] Inference error: {e}")
        return []


def extract_with_groq(testimony: str, speaker: str) -> list:
    prompt = (
        f"Extract structured facts from this murder investigation testimony.\n"
        f"Speaker: {speaker}\nTestimony: {testimony[:400]}\n\n"
        f"Return a JSON array. Each item: subject, predicate "
        f"(one of: was_at was_with has_motive owns knew did_action), "
        f"obj, time (or null), confidence.\n"
        f"Return ONLY the JSON array."
    )
    raw = call_groq([{"role": "user", "content": prompt}], max_tokens=300)
    return _parse_json_facts(raw, speaker, testimony)


print("Extractor functions ready.")

In [ ]:
class KnowledgeStore:

    def __init__(self):
        self.facts             : list[Fact]                = []
        self.contradictions    : list[Contradiction]       = []
        self.suspects          : dict[str, SuspectProfile] = {}
        self.interrogation_log : list[dict]                = []

    def add_suspect(self, name: str):
        if name not in self.suspects:
            self.suspects[name] = SuspectProfile(name=name)

    def add_fact(self, fact: Fact):
        self.facts.append(fact)
        if fact.subject in self.suspects:
            self.suspects[fact.subject].facts.append(fact)

    def add_contradiction(self, c: Contradiction):
        self.contradictions.append(c)
        for subj in (c.fact1.subject, c.fact2.subject):
            if subj in self.suspects:
                self.suspects[subj].contradictions.append(c)

    def log_qa(self, suspect: str, question: str, answer: str):
        self.interrogation_log.append({
            "suspect" : suspect,
            "question": question,
            "answer"  : answer,
        })

    def facts_about(self, subject: str) -> list[Fact]:
        return [f for f in self.facts if f.subject == subject]

    def facts_by_predicate(self, predicate: str) -> list[Fact]:
        return [f for f in self.facts if f.predicate == predicate]

    def alibi_facts(self) -> list[Fact]:
        return self.facts_by_predicate("was_at")

    def get_suspect_names(self) -> list[str]:
        return list(self.suspects.keys())


class FactExtractor:

    def extract(self, testimony: str, speaker: str) -> list[Fact]:
        if not testimony.strip():
            return []
        if FINETUNED_AVAILABLE:
            facts = extract_with_finetuned(testimony, speaker)
            if facts:
                return facts
        return extract_with_groq(testimony, speaker)


class ContradictionDetector:

    def __init__(self, store: KnowledgeStore):
        self.store = store

    def run(self) -> list[Contradiction]:
        found = []
        found.extend(self._location_conflicts())
        found.extend(self._mutual_witness_conflicts())
        found.extend(self._motive_denial_conflicts())
        return found

    def _location_conflicts(self) -> list[Contradiction]:
        alibi = self.store.alibi_facts()
        found = []
        for i, f1 in enumerate(alibi):
            for f2 in alibi[i + 1:]:
                if (
                    f1.subject == f2.subject and
                    f1.source  != f2.source  and
                    f1.obj.lower() != f2.obj.lower()
                ):
                    exp = (
                        f"{f1.subject} cannot simultaneously be at "
                        f"'{f1.obj}' ({f1.source}) and "
                        f"'{f2.obj}' ({f2.source})."
                    )
                    found.append(Contradiction(f1, f2, "location_conflict", exp, 0.9))
        return found

    def _mutual_witness_conflicts(self) -> list[Contradiction]:
        found       = []
        was_with    = self.store.facts_by_predicate("was_with")
        alone_facts = [
            f for f in self.store.facts_by_predicate("did_action")
            if "alone" in f.obj.lower()
        ]
        for wf in was_with:
            for af in alone_facts:
                if wf.obj == af.subject and wf.source != af.source:
                    exp = (
                        f"{wf.source} claims to have been with "
                        f"{wf.obj}, but {af.subject} claims to have been alone."
                    )
                    found.append(Contradiction(wf, af, "witness_conflict", exp, 0.8))
        return found

    def _motive_denial_conflicts(self) -> list[Contradiction]:
        found = []
        for mf in self.store.facts_by_predicate("has_motive"):
            for kf in self.store.facts_by_predicate("knew"):
                if mf.subject == kf.subject and kf.confidence < 0.5:
                    exp = (
                        f"{mf.subject} has a documented motive but "
                        f"appears to deny close acquaintance with the victim."
                    )
                    found.append(Contradiction(mf, kf, "motive_denial", exp, 0.75))
        return found

    def get_new(self) -> list[Contradiction]:
        existing = {(c.fact1.raw, c.fact2.raw) for c in self.store.contradictions}
        return [
            c for c in self.run()
            if (c.fact1.raw, c.fact2.raw) not in existing
        ]


class SuspicionScorer:

    def __init__(self, store: KnowledgeStore):
        self.store     = store
        self.score_log : list[dict] = []

    def update_contradiction(self, c: Contradiction):
        delta = W_CONTRADICTION * c.confidence
        self._apply(c.fact1.subject, delta,
                    f"Alibi contradiction ({c.kind}): {c.explanation}")

    def update_motive(self, suspect: str):
        if not self.store.suspects[suspect].has_motive:
            self.store.suspects[suspect].has_motive = True
            self._apply(suspect, W_MOTIVE, "Established motive against victim")

    def update_no_alibi(self, suspect: str):
        self._apply(suspect, W_NO_ALIBI, "No verifiable alibi for the murder window")

    def update_verified_alibi(self, suspect: str):
        self.store.suspects[suspect].alibi_verified = True
        self._apply(suspect, W_ALIBI_VERIFIED, "Alibi independently corroborated")

    def _apply(self, suspect: str, delta: float, reason: str):
        if suspect not in self.store.suspects:
            return
        profile   = self.store.suspects[suspect]
        old_score = profile.suspicion_score
        new_score = old_score + delta
        profile.suspicion_score = new_score
        self.score_log.append({
            "suspect"  : suspect,
            "delta"    : round(delta, 2),
            "reason"   : reason,
            "old_score": round(old_score, 2),
            "new_score": round(new_score, 2),
        })
        print(f"[Score] {suspect}: {old_score:.1f} -> {new_score:.1f} | {reason}")

    def ranking(self) -> list[tuple]:
        return sorted(
            [(n, p.suspicion_score) for n, p in self.store.suspects.items()],
            key=lambda x: x[1], reverse=True,
        )

    def top_suspect(self) -> Optional[str]:
        r = self.ranking()
        return r[0][0] if r else None


class EvidencePriorityQueue:

    def __init__(self, store: KnowledgeStore):
        self.store = store

    def compute_priority(self, name: str) -> float:
        profile = self.store.suspects.get(name)
        if not profile:
            return 0.0
        contradiction_score = len(profile.contradictions) * 25
        motive_score        = 20 if profile.has_motive else 0
        no_alibi_score      = 15 if not profile.alibi_verified else 0
        fatigue_penalty     = profile.questions_asked * 3
        return contradiction_score + motive_score + no_alibi_score - fatigue_penalty

    def next_suspect(self) -> Optional[str]:
        scores = {
            name: self.compute_priority(name)
            for name in self.store.get_suspect_names()
        }
        if not scores:
            return None
        max_score    = max(scores.values())
        top_suspects = [n for n, s in scores.items() if s == max_score]
        return random.choice(top_suspects)


class InterrogatorAgent:

    BASELINE_QUESTIONS = [
        "Where were you on the evening of the murder?",
        "What was your relationship with the victim?",
        "Did you notice anything unusual that evening?",
    ]

    DEEP_QUESTIONS = [
        "Did you have any reason to want the victim dead?",
        "Can anyone confirm your whereabouts that evening?",
        "Do you know who might have wanted to harm the victim?",
    ]

    def __init__(self, store: KnowledgeStore, victim: str):
        self.store    = store
        self.victim   = victim
        self.histories: dict[str, list] = {}

    def initialize(self, name: str, profile: dict):
        intro        = normalize(profile.get("introduction", ""))[:250]
        relationship = normalize(profile.get("relationship",  ""))[:150]
        suspicion    = normalize(profile.get("suspicion",     ""))[:150]
        system = (
            f"You are {name}, a suspect in a murder investigation. "
            f"Background: {intro} "
            f"Your relationship with the victim: {relationship} "
            f"Additional context: {suspicion} "
            f"Stay in character at all times. "
            f"Answer in 3-4 sentences. "
            f"Never acknowledge being an AI."
        )
        self.histories[name] = [
            {"role": "user",      "content": system},
            {"role": "assistant", "content": f"Understood. I am {name}."},
        ]

    def ask(self, suspect: str, question: str) -> str:
        if suspect not in self.histories:
            print(f"[Interrogator] {suspect} was not initialised.")
            return ""
        self.histories[suspect].append({"role": "user", "content": question})
        answer = call_groq(self.histories[suspect], max_tokens=200)
        if answer:
            self.histories[suspect].append({"role": "assistant", "content": answer})
        self.store.log_qa(suspect, question, answer)
        if suspect in self.store.suspects:
            self.store.suspects[suspect].questions_asked += 1
        print(f"\n[Det -> {suspect}]: {question}")
        print(f"[{suspect}]: {answer}")
        return answer

    def confront(self, suspect: str, c: Contradiction) -> str:
        q = (
            f"I have conflicting information. {c.explanation} "
            f"How do you explain this discrepancy?"
        )
        return self.ask(suspect, q)


print("All system classes defined.")

In [ ]:
class InvestigativeLoop:

    def __init__(self, case_data: dict):
        self.case_data    = case_data
        self.victim       = case_data["victim"]["name"]
        self.location     = case_data["location"]
        self.time         = case_data["time"]

        self.store        = KnowledgeStore()
        self.extractor    = FactExtractor()
        self.detector     = ContradictionDetector(self.store)
        self.scorer       = SuspicionScorer(self.store)
        self.interrogator = InterrogatorAgent(self.store, self.victim)
        self.pq           = EvidencePriorityQueue(self.store)
        self.decision_log : list[dict] = []

    def _log(self, decision: str, reason: str):
        self.decision_log.append({"decision": decision, "reason": reason})
        print(f"\n[Decision] {decision}")
        print(f"[Reason]   {reason}")

    def _absorb(self, testimony: str, speaker: str):
        for fact in self.extractor.extract(testimony, speaker):
            self.store.add_fact(fact)

    def _score_profile(self, name: str, profile: dict):
        combined = (
            normalize(profile.get("suspicion", "")) + " " +
            normalize(profile.get("motive",    ""))
        ).lower()
        if any(kw in combined for kw in MOTIVE_KEYWORDS):
            self.scorer.update_motive(name)
        for f in self.store.facts_about(name):
            if f.predicate == "has_motive":
                self.scorer.update_motive(name)
                break
        self_alibi = [
            f for f in self.store.facts_about(name)
            if f.predicate == "was_at" and f.source == name
        ]
        if not self_alibi:
            self.scorer.update_no_alibi(name)

    def _termination_check(self) -> bool:
        ranking = self.scorer.ranking()
        if len(ranking) < 2:
            return False
        top_name,  top_score = ranking[0]
        _,         sec_score = ranking[1]
        if sec_score == 0:
            return False
        gap_met  = top_score >= sec_score * VERDICT_SCORE_GAP_RATIO
        profile  = self.store.suspects[top_name]
        evidence = profile.has_motive or len(profile.contradictions) > 0
        return gap_met and evidence

    def _phase1(self):
        print("\n=== PHASE 1: INITIAL SWEEP ===")
        for s in self.case_data["suspects"]:
            name = s["name"]
            self._log(
                f"Baseline interrogation: {name}",
                "Every suspect must account for their alibi and relationship "
                "with the victim before targeted questioning begins.",
            )
            for q in self.interrogator.BASELINE_QUESTIONS:
                ans = self.interrogator.ask(name, q)
                self._absorb(ans, name)
            self._score_profile(name, s)

    def _phase2(self):
        print("\n=== PHASE 2: CONTRADICTION HUNTING ===")
        for rnd in range(MAX_CONTRADICTION_ROUNDS):
            print(f"\n[Round {rnd + 1}]")
            new_contradictions = self.detector.get_new()
            if not new_contradictions:
                self._log(
                    "No new contradictions detected",
                    "Contradiction hunting phase concluded early.",
                )
                break
            for c in new_contradictions:
                self.store.add_contradiction(c)
                self.scorer.update_contradiction(c)
                self._log(
                    f"Confronting {c.fact1.subject} with contradiction",
                    c.explanation,
                )
                ans = self.interrogator.confront(c.fact1.subject, c)
                self._absorb(ans, c.fact1.subject)
            if self._termination_check():
                self._log(
                    "Early termination triggered",
                    "Sufficient evidence gap established after contradiction round.",
                )
                return

    def _phase3(self):
        print("\n=== PHASE 3: TARGETED PRESSURE ===")
        ranking      = self.scorer.ranking()
        top_suspects = [n for n, sc in ranking[:2] if sc > 0]
        for name in top_suspects:
            priority = self.pq.compute_priority(name)
            self._log(
                f"Targeted pressure: {name}",
                f"Priority score = {priority:.1f} | "
                f"Suspicion = {self.store.suspects[name].suspicion_score:.1f}",
            )
            for q in self.interrogator.DEEP_QUESTIONS:
                ans = self.interrogator.ask(name, q)
                self._absorb(ans, name)
            if self._termination_check():
                self._log(
                    "Early termination triggered",
                    "Sufficient evidence gap established during targeted pressure phase.",
                )
                return

    def _conclude(self) -> dict:
        print("\n=== CONCLUSION ===")
        ranking = self.scorer.ranking()
        if not ranking:
            return {
                "verdict": "inconclusive", "suspect": None,
                "confidence": "none", "score": 0, "ranking": [],
            }
        top_name,  top_score = ranking[0]
        _,         sec_score = ranking[1] if len(ranking) > 1 else (None, 0)
        profile      = self.store.suspects[top_name]
        has_motive   = profile.has_motive
        alibi_broken = len(profile.contradictions) > 0
        score_gap    = (
            top_score >= sec_score * VERDICT_SCORE_GAP_RATIO
            if sec_score > 0 else top_score > 0
        )
        self._log(
            f"Verdict evaluation: {top_name}",
            f"score={top_score:.1f}, second={sec_score:.1f}, "
            f"motive={has_motive}, alibi_broken={alibi_broken}, "
            f"score_gap_met={score_gap}",
        )
        if top_score > 0 and score_gap:
            verdict    = "guilty"
            confidence = (
                "high"   if (has_motive and alibi_broken) else
                "medium" if (has_motive or  alibi_broken) else
                "low"
            )
        elif top_score > 0:
            verdict    = "guilty"
            confidence = "low"
        else:
            verdict    = "inconclusive"
            confidence = "none"
        self._log(
            f"Final verdict: {top_name} - {verdict}",
            f"Confidence: {confidence}",
        )
        return {
            "verdict"   : verdict,
            "suspect"   : top_name,
            "confidence": confidence,
            "score"     : top_score,
            "ranking"   : ranking,
        }

    def run(self) -> dict:
        print("\n" + "=" * 55)
        print(f"CASE  : {self.location}")
        print(f"VICTIM: {self.victim}")
        print(f"TIME  : {self.time}")
        print("=" * 55)
        for s in self.case_data["suspects"]:
            self.store.add_suspect(s["name"])
            self.interrogator.initialize(s["name"], s)
        self._phase1()
        self._phase2()
        self._phase3()
        result = self._conclude()
        result.update({
            "decision_log"     : self.decision_log,
            "interrogation_log": self.store.interrogation_log,
            "score_log"        : self.scorer.score_log,
            "facts"            : self.store.facts,
            "contradictions"   : self.store.contradictions,
        })
        return result


print("InvestigativeLoop defined.")

In [ ]:
class ReportGenerator:

    def __init__(self, case_data: dict, result: dict):
        self.case   = case_data
        self.result = result

    def generate(self) -> str:
        sections = [
            self._header(),
            self._verdict(),
            self._reconstruction(),
            self._evidence(),
            self._scores(),
            self._timeline(),
            self._decisions(),
            self._errors(),
        ]
        return "\n\n".join(sections)

    def _header(self) -> str:
        suspects = ", ".join(s["name"] for s in self.case["suspects"])
        return (
            f"{'=' * 55}\n"
            f"CASE REPORT\n"
            f"{'=' * 55}\n"
            f"Location : {self.case['location']}\n"
            f"Time     : {self.case['time']}\n"
            f"Victim   : {self.case['victim']['name']}\n"
            f"Suspects : {suspects}\n"
            f"{'=' * 55}"
        )

    def _verdict(self) -> str:
        v = self.result.get("verdict", "inconclusive")
        if v == "guilty":
            return (
                f"--- VERDICT ---\n"
                f"Murderer   : {self.result['suspect']}\n"
                f"Confidence : {self.result['confidence']}\n"
                f"Score      : {self.result['score']:.1f}"
            )
        return "--- VERDICT ---\nInconclusive. Insufficient evidence to name a murderer."

    def _reconstruction(self) -> str:
        lines     = ["--- CHRONOLOGICAL RECONSTRUCTION ---"]
        movements = [f for f in self.result.get("facts", []) if f.predicate == "was_at"]
        if movements:
            lines.append("Suspect movements on the night of the crime:")
            for m in movements:
                t = m.time or "unspecified time"
                lines.append(f"  {m.subject} was at {m.obj} ({t}) - reported by {m.source}")
        else:
            lines.append("No location facts extracted.")
        contradictions = self.result.get("contradictions", [])
        if contradictions:
            lines.append("\nConflicting accounts:")
            for c in contradictions:
                lines.append(f"  [{c.kind}] {c.explanation}")
        suspect = self.result.get("suspect")
        if suspect:
            profile_facts = [f for f in self.result.get("facts", []) if f.subject == suspect]
            lines.append(f"\nKey evidence against {suspect}:")
            for f in profile_facts[:5]:
                lines.append(f"  {f.predicate}: {f.obj} (source: {f.source})")
        return "\n".join(lines)

    def _evidence(self) -> str:
        lines = ["--- EVIDENCE SUMMARY ---"]
        cs    = self.result.get("contradictions", [])
        lines.append(f"Total contradictions : {len(cs)}")
        for c in cs:
            lines.append(f"  [{c.kind}] conf={c.confidence:.2f} | {c.explanation}")
        fs = self.result.get("facts", [])
        lines.append(f"\nTotal facts extracted: {len(fs)}")
        for f in fs:
            lines.append(
                f"  {f.subject} | {f.predicate} | {f.obj} "
                f"| time={f.time} | src={f.source} | conf={f.confidence:.2f}"
            )
        return "\n".join(lines)

    def _scores(self) -> str:
        lines = ["--- SUSPICION SCORE PROGRESSION ---"]
        for e in self.result.get("score_log", []):
            lines.append(
                f"  {e['suspect']}: "
                f"{e['old_score']:.1f} -> {e['new_score']:.1f} "
                f"| {e['reason']}"
            )
        lines.append("\nFinal Ranking:")
        for name, sc in self.result.get("ranking", []):
            lines.append(f"  {name}: {sc:.1f}")
        return "\n".join(lines)

    def _timeline(self) -> str:
        lines = ["--- INTERROGATION TIMELINE ---"]
        for i, e in enumerate(self.result.get("interrogation_log", []), 1):
            lines.append(
                f"\nQ{i} [{e['suspect']}]\n"
                f"  Detective : {e['question']}\n"
                f"  {e['suspect']}: {e['answer']}"
            )
        return "\n".join(lines)

    def _decisions(self) -> str:
        lines = ["--- DECISION TRAIL ---"]
        for i, e in enumerate(self.result.get("decision_log", []), 1):
            lines.append(
                f"\nDecision {i}: {e['decision']}\n"
                f"  Reason: {e['reason']}"
            )
        return "\n".join(lines)

    def _errors(self) -> str:
        return "--- ERRORS / ANOMALIES ---\nNone recorded."

    def save(self, path: str) -> str:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        report = self.generate()
        with open(path, "w", encoding="utf-8") as f:
            f.write(report)
        return report


print("ReportGenerator defined.")

In [ ]:
token_usage   = {"prompt": 0, "completion": 0, "total": 0}
eval_start    = time.time()
EVAL_CASES    = dataset[300:350]
results_log   = []

completed_cases = set()
for fname in os.listdir("outputs"):
    if fname.startswith("case_") and fname.endswith(".json"):
        try:
            cid = int(fname.split("_")[1].split(".")[0])
            completed_cases.add(cid)
        except Exception:
            pass

for i, case in enumerate(EVAL_CASES):
    actual_case_id = 300 + i
    if actual_case_id in completed_cases:
        print(f"Skipping completed case {actual_case_id}")
        continue

    print(f"\n{'=' * 55}")
    print(f"Case {i + 1}/{len(EVAL_CASES)}: {case['location']}")
    print(f"{'=' * 55}")

    case_start = time.time()

    try:
        loop   = InvestigativeLoop(case)
        result = loop.run()

        actual = next(
            (s["name"] for s in case["suspects"] if s.get("is_murderer") is True),
            "Unknown",
        )
        our     = result.get("suspect") or "inconclusive"
        correct = our == actual
        elapsed = time.time() - case_start

        case_result = {
            "case"             : case["location"],
            "our_verdict"      : our,
            "actual"           : actual,
            "correct"          : correct,
            "confidence"       : result.get("confidence"),
            "score"            : result.get("score", 0),
            "n_facts"          : len(result.get("facts", [])),
            "n_contradictions" : len(result.get("contradictions", [])),
            "time_seconds"     : round(elapsed, 1),
        }
        results_log.append(case_result)

        with open(os.path.join("outputs", f"case_{actual_case_id}.json"), "w") as f:
            json.dump(case_result, f, indent=2)

        ReportGenerator(case, result).save(
            os.path.join(REPORTS_DIR, f"case_{actual_case_id}.txt")
        )

        print(f"\nOUR VERDICT : {our}")
        print(f"ACTUAL      : {actual}")
        print(f"RESULT      : {'CORRECT' if correct else 'WRONG'}")
        print(f"TIME        : {elapsed:.1f}s")

    except Exception as e:
        print(f"[ERROR] Case {i + 1} failed: {e}")
        with open(os.path.join("logs", f"error_case_{actual_case_id}.json"), "w") as f:
            json.dump({"case_id": actual_case_id, "error": str(e)}, f, indent=2)
        results_log.append({
            "case": case.get("location", "unknown"), "our_verdict": "error",
            "actual": "unknown", "correct": False, "confidence": None,
            "score": 0, "n_facts": 0, "n_contradictions": 0, "time_seconds": 0,
        })
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

total_elapsed = time.time() - eval_start
print(f"\nEvaluation complete in {total_elapsed / 60:.1f} minutes.")

In [ ]:
correct_count = sum(1 for r in results_log if r["correct"])
total_count   = len(results_log)
accuracy      = correct_count / total_count * 100 if total_count else 0

avg_facts = sum(r["n_facts"]          for r in results_log) / total_count
avg_contr = sum(r["n_contradictions"] for r in results_log) / total_count
avg_time  = sum(r["time_seconds"]     for r in results_log) / total_count

print("\n" + "=" * 55)
print("FINAL EVALUATION RESULTS")
print("=" * 55)
print(f"Cases evaluated      : {total_count}")
print(f"Correct verdicts     : {correct_count}")
print(f"Accuracy             : {accuracy:.1f}%")
print(f"Avg facts / case     : {avg_facts:.1f}")
print(f"Avg contradictions   : {avg_contr:.1f}")
print(f"Avg time / case      : {avg_time:.1f}s")
print(f"\nToken usage (Groq):")
print(f"  Prompt tokens      : {token_usage['prompt']:,}")
print(f"  Completion tokens  : {token_usage['completion']:,}")
print(f"  Total tokens       : {token_usage['total']:,}")
print(f"  Tokens / case      : {token_usage['total'] // total_count:,}")

print("\nPer-case breakdown:")
for r in results_log:
    status = "CORRECT" if r["correct"] else "WRONG"
    print(
        f"  [{status}] {r['case'][:38]:<38} "
        f"pred={r['our_verdict'][:20]:<20} "
        f"actual={r['actual']}"
    )

summary = {
    "total_cases"        : total_count,
    "correct"            : correct_count,
    "accuracy_pct"       : round(accuracy, 2),
    "avg_facts"          : round(avg_facts, 2),
    "avg_contradictions" : round(avg_contr, 2),
    "avg_time_seconds"   : round(avg_time, 2),
    "token_usage"        : token_usage,
    "per_case"           : results_log,
}
with open("evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nEvaluation summary saved to evaluation_summary.json")
print(f"Individual case reports saved to {REPORTS_DIR}/")